In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig, VitsModel
import time
import psutil
import torch
import whisper
import sounddevice as sd
import numpy as np
from IPython.display import Audio

c:\Users\kimbe\Documents\GitHub\kata-ondevice\kata-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- GEMMA ---

model_id = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = QuantoConfig(weights="int8")

model = AutoModelForCausalLM.from_pretrained(model_id,
    device_map="cpu",
    quantization_config=quantization_config
    )

Loading checkpoint shards: 100%|██████████| 2/2 [00:23<00:00, 11.79s/it]


In [3]:
# --- STT ---

whisper_model = whisper.load_model("small")

def perintah():
    duration = 5
    sample_rate = 16000  

    print("Mendengarkan......")
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()  
    print("Diterima.....")

    audio_data = np.squeeze(audio_data)  
    dengar = whisper_model.transcribe(audio_data, fp16=False, language="id")
    print(dengar["text"])

    return dengar["text"]

c:\Users\kimbe\Documents\GitHub\kata-ondevice\kata-venv\Lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, 

In [4]:
# --- TTS ---

mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
mms_token = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

def ngomong(text):
    inputs = mms_token(text, return_tensors="pt")

    with torch.no_grad():
        output = mms(**inputs).waveform
        return Audio(output.squeeze().cpu().numpy(), rate=16000) # Can change rate to make it faster/slower

In [5]:
# --- MAIN FUNCTION ---

def run_va():
    Layanan = perintah()
    
    system_prompt= 'Jawab dalam bahasa indonesia.'

    messages = [
    {"role": "user", "content": Layanan}
]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt") 

    start_time = time.time()
    memory_before = psutil.virtual_memory().used

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=512, 
        temperature=0.7,
        top_p=0.9,
        do_sample=True

    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    end_time = time.time()
    memory_after = psutil.virtual_memory().used

    inference_time = end_time - start_time
    memory_used = memory_after - memory_before
    cpu_usage = psutil.cpu_percent(interval=1)
    
    print(response)
    print("INFERENCE TIME: ", inference_time)
    print("MEMORY USAGE: ", memory_used)
    print("CPU USAGE: ", cpu_usage)
    ngomong(response)

In [6]:
run_va()

Mendengarkan......
Diterima.....
 Apa Ibu Krota Indonesia?


KeyboardInterrupt: 

In [7]:
input_text = "Apa ibukota Indonesia?"
input_ids = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(**input_ids, max_new_tokens=32)
print(tokenizer.decode(outputs[0]))


<bos>Apa ibukota Indonesia?

Jawabannya adalah **Jakarta**. 
<end_of_turn>
